# Import the necessary libraries and modules

In [1]:
import torch  # PyTorch library for deep learning  
import torch.nn as nn  # Neural network modules and loss functions  
import pennylane as qml  # PennyLane for quantum machine learning  
import numpy as np  # NumPy library for numerical computations  
import matplotlib.pyplot as plt  # Plotting library for visualization  
import pandas as pd  # Library for data manipulation and analysis  
dtype = torch.float  # Set default tensor data type to float


In [2]:
def Solution(t, params):
    # Analytical solution of the damped harmonic oscillator for comparison
    A, omega, beta = params
    return A*torch.exp(-beta*t)*torch.cos(np.sqrt((omega**2) - (beta**2))*t)
    
def test_PINN(NN, domain, params, num_test):
    # Evaluate the neural network against the analytical solution and compute error metrics
    t = torch.linspace(domain[0], domain[1], num_test).view(-1, 1)
    pred_solution = NN(t)
    exact_solution = Solution(t, params)
    
    l2_error = (exact_solution - pred_solution)**2
    l2_metric = torch.mean(l2_error)

    return l2_metric

In [3]:
# Set the necessary parameters
A , k, m, b = 0.5, 9, 1, 1.6
omega = np.sqrt(k/m)
beta = b / (2*m)
params = [A, omega, beta]
domain, num_test = [0.0, 8.0], 500

# Classical Neural Network

In [4]:
classic_model_list = []
classic_history_lists = []

def init_weights(m):
    """
    Initializes weights and biases for a linear layer.

    Args:
        m (nn.Linear): The linear layer to initialize.

    Notes:
        - Applies uniform weight initialization within the range [-n0, n0].
        - Initializes biases with a constant value of 0.01.
    """
    if isinstance(m, nn.Linear):
        out_sz = torch.tensor(m.out_features)
        n0 = 1.0/torch.sqrt(out_sz)
        m.weight.data.uniform_(-n0, n0)
        # m.bias.data.fill_(0.01)

class CNeuralNet(nn.Module):
    """
    Classical neural network class.

    Args:
        input_size (int): Size of the input features.
        hidden_size (int): Number of hidden units in the neural network.
        output_size (int): Size of the output.

    Attributes:
        fnn (nn.Sequential): Feedforward neural network with three linear layer followed by a Silu activation.

    Notes:
        - The classical neural network architecture consists of three hidden layers.
        - The input layer has 'input_size' neurons.
        - The output layer has 'output_size' neurons.
    """
    def __init__(self, input_size, hidden_size, output_size):
        super(CNeuralNet, self).__init__()
        self.fnn = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, output_size),
        )

        self.fnn.apply(init_weights)

    def forward(self, x):
        """
        Forward pass through the neural network.

        Args:
            x (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        out = self.fnn(x)
        out = out[:, 0]*out[:, 1]
        out = out.view(-1, 1)
        return out

In [5]:
# Your seeds and configs
seeds = [14, 42, 86, 195]
hidden_sizes = [10, 30, 50]

# Domain and params used in test_PINN

# Store results for table
results = []

for hs in hidden_sizes:
    mses = []

    for seed in seeds:
        # Load model
        input_size, hidden_size, output_size = 1, hs, 2
        model_path = f"..\\trained_models\\classical damped harmonic oscillator\\{seed}\\Classical_model_{seed}_{hs}.pth"
        model = CNeuralNet(input_size, hidden_size, output_size)
        model.load_state_dict(torch.load(model_path))
        model.eval()

        # Evaluate model
        with torch.no_grad():
            mse = test_PINN(model, domain, params, num_test)
            mses.append(mse.item())

    # Compute stats
    mean_mse = np.mean(mses)
    std_mse = np.std(mses)

    # Add to results
    results.append([hs, mean_mse, std_mse])

# Create table
df = pd.DataFrame(results, columns=["Hidden Size", "Mean MSE", "Std. Dev. MSE"])

# Format numbers in scientific notation
df["Mean MSE"] = df["Mean MSE"].map(lambda x: f"{x:.5e}")
df["Std. Dev. MSE"] = df["Std. Dev. MSE"].map(lambda x: f"{x:.5e}")

print(df.to_string(index=False))


 Hidden Size    Mean MSE Std. Dev. MSE
          10 4.16606e-06   4.67974e-06
          30 7.59230e-07   2.83447e-07
          50 1.40725e-06   1.24004e-06


# Quantum Neural Network

In [6]:
# Set the number of qubits to 3
n_qubits = 2
# Create a quantum device with 3 qubits
dev = qml.device("default.qubit", wires=n_qubits)

def RY_layer(w):
    """
    Apply a layer of single-qubit rotations around the Y-axis (RY gates).

    Args:
        w (torch.Tensor): List of rotation angles for each qubit.
    """
    for idx, element in enumerate(w):
        qml.RY(element, wires=idx)

def entangling_layer(nqubits):
    """
    Apply an entangling layer of CNOT gates to adjacent qubits.

    Args:
        nqubits (int): Number of qubits in use.
    """
    for i in range(0, nqubits - 1):
        qml.CNOT(wires=[i, i + 1])

@qml.qnode(dev)
def quantum_net(input_features, n_qubits, ansatz_depth, feature_weights, ansatz_weights, out_weights):
    """
    Quantum circuit

    Args:
        input_features (torch.Tensor): Input features.
        n_qubits (int): Number of qubits in the circuit.
        ansatz_depth (int): Depth of the ansatz (number of layers).
        feature_weights (torch.Tensor): List of parameters
        ansatz_weights (torch.Tensor): List of parameters
        out_weights (torch.Tensor): List of parameters

    Returns:
        tuple: Tuple of expectation values for Pauli-Z operators on qubits 0 and 1 as the solution.
    """
    # apply feature map layers
    temp1 = feature_weights[0]*input_features[0]
    temp2 = feature_weights[1]*input_features[0]
    qml.exp(qml.PauliZ(0), temp1[0])
    qml.exp(qml.PauliZ(1), temp2[0])
    entangling_layer(n_qubits)
    temp3 = feature_weights[2]*(input_features[0])
    temp4 = feature_weights[3]*(input_features[0])
    qml.RY(temp3, wires=0)
    qml.RY(temp4, wires=1)
    entangling_layer(n_qubits)

    # apply ansatz layers
    for k in range(ansatz_depth):
        RY_layer(ansatz_weights[k])
        entangling_layer(n_qubits)

    # Compute expectation values for Pauli-Z operators
    exp_vals = [qml.expval(out_weights[0]*(qml.PauliZ(0)@qml.PauliZ(1)))]
    return tuple(exp_vals)


class QNeuralNet(nn.Module):
    """
    quantum neural network class.

    Args:
        n_qubits (int): Number of qubits in the circuit.
        feature_depth (int): Depth of the feature map (number of layers).
        ansatz_depth (int): Depth of the ansatz (number of layers).

    Notes:
        - The quantum neural network architecture consists of two qubits.
        - The feature map has 'feature_depth' layers.
        - The ansatz has 'ansatz_depgh' layers.
    """
    def __init__(self, n_qubits, feature_depth, ansatz_depth):
        super(QNeuralNet, self).__init__()

        # Set the number qubits, number of ansatz layer and feature map layers
        self.n_qubits = n_qubits
        self.ansatz_depth = ansatz_depth
        self.feature_depth = feature_depth
        
        # Initialize parameters of quantum circuit
        self.feature_weights = nn.Parameter(0.1*torch.randn((feature_depth*n_qubits + 2)).view(-1, 1))
        self.ansatz_weights = nn.Parameter(torch.randn(ansatz_depth*n_qubits).view(ansatz_depth, n_qubits))
        self.out_weights = nn.Parameter(torch.randn((1)))

    def forward(self, x):
        """
        Forward pass through the neural network.

        Args:
            x (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        # Initialize an empty tensor for quantum output
        q_out = torch.Tensor(0, 1)

        for elem in x:
            # Apply the quantum circuit (quantum_net) to each input element
            q_out_elem = torch.hstack(tuple(quantum_net(elem, self.n_qubits, self.ansatz_depth, self.feature_weights,self.ansatz_weights, self.out_weights))).float().unsqueeze(0)
            q_out = torch.cat((q_out, q_out_elem)) # Concatenate quantum outputs

        return q_out

In [7]:
# Define seeds and ansatz depths (i.e., quantum layers)
seeds = [14, 42, 86, 195]
ansatz_layers = [1, 2, 3]

# Assume fixed n_qubits and feature_depth
n_qubits = 2  # Update this as needed
feature_depth = 1


# Store results
results = []

for layer in ansatz_layers:
    mses = []

    for seed in seeds:
        # Initialize and load model
        ansatz_depth = layer
        model_path = f"..\\trained_models\\quantum damped harmonic oscillator\\{seed}\\Quantum_model_{seed}_{layer}.pth"
        model = QNeuralNet(n_qubits, feature_depth, ansatz_depth)
        model.load_state_dict(torch.load(model_path))
        model.eval()

        # Evaluate model
        with torch.no_grad():
            mse = test_PINN(model, domain, params, num_test)
            mses.append(mse.item())

    # Compute statistics
    mean_mse = np.mean(mses)
    std_mse = np.std(mses)

    # Add result
    results.append([layer, mean_mse, std_mse])

# Create DataFrame
df = pd.DataFrame(results, columns=["Ansatz Depth", "Mean MSE", "Std. Dev. MSE"])

# Format for scientific notation
df["Mean MSE"] = df["Mean MSE"].map(lambda x: f"{x:.5e}")
df["Std. Dev. MSE"] = df["Std. Dev. MSE"].map(lambda x: f"{x:.5e}")

print(df.to_string(index=False))


 Ansatz Depth    Mean MSE Std. Dev. MSE
            1 2.04924e-11   2.92058e-11
            2 9.79263e-11   1.35941e-10
            3 2.56206e-09   2.64516e-09


# Hybrid Quantum Neural Network

In [8]:
# Set the number of qubits to 3
n_qubits = 3
# Create a quantum device with 3 qubits
dev = qml.device("default.qubit", wires=n_qubits)

def RY_layer(w):
    """
    Apply a layer of single-qubit rotations around the Y-axis (RY gates).

    Args:
        w (torch.Tensor): List of rotation angles for each qubit.
    """
    for idx, element in enumerate(w):
        qml.RY(element, wires=idx)

def entangling_layer(nqubits):
    """
    Apply an entangling layer of CNOT gates to adjacent qubits.

    Args:
        nqubits (int): Number of qubits in use.
    """
    for i in range(0, nqubits - 1):
        qml.CNOT(wires=[i, i + 1])

def init_weights(m):
    """
    Initializes weights and biases for a linear layer.

    Args:
        m (nn.Linear): The linear layer to initialize.

    Notes:
        - Applies uniform weight initialization within the range [-n0, n0].
        - Initializes biases with a constant value of 0.01.
    """
    if isinstance(m, nn.Linear):
        out_sz = torch.tensor(m.out_features)
        n0 = 1.0/torch.sqrt(out_sz)
        m.weight.data.uniform_(-n0, n0)
        # m.bias.data.fill_(0.01)

@qml.qnode(dev)
def hquantum_net(input_features, n_qubits, q_depth, q_weights):
    """
    Quantum circuit as a part of quantum hybrid neural network.

    Args:
        input_features (torch.Tensor): Input features.
        n_qubits (int): Number of qubits in the circuit.
        q_depth (int): Number of quantum layers.
        q_weights (torch.Tensor): List of parameters.
    Returns:
        tuple: Tuple of expectation values for Pauli-Z operators on qubits 0, 1, and 2.
    """

    RY_layer(input_features[0])

    entangling_layer(n_qubits)
    RY_layer(input_features[1])
    RY_layer(q_weights[0])

    entangling_layer(n_qubits)
    RY_layer(input_features[2])
    RY_layer(q_weights[1])

    entangling_layer(n_qubits)
    RY_layer(input_features[3])
    RY_layer(q_weights[2])

    exp_vals = [qml.expval(qml.PauliZ(position)) for position in range(0, n_qubits)]
    return tuple(exp_vals)

class HQNeuralNet(nn.Module):
    """
    quantum hybrid neural network class.

    Args:
        input_size (int): Size of the input features.
        hidden_size (int): Number of hidden units in the neural network.
        n_qubits (int): Number of qubits in the circuit.
        q_depth (int): Number of quantum layers.
        output_size (int): Size of the output.

    Attributes:
        fnn (nn.Sequential): Feedforward neural network with a linear layer followed by a Tanh activation.

    Notes:
        - The first classical neural network architecture consists of one hidden layer.
        - The input layer has 'input_size' neurons.
        - The hidden layers has 'hidden_size' neurons with a Tanh activation.
        - The quantum neural network architecture consists of 'n_qubits' and 'q_depth ' layers.
        - The second classical neural network architecture consists of one linear layer.
        - The output layer has 'output_size' neurons.
    """
    def __init__(self, input_size, hidden_size, n_qubits, q_depth, output_size):
        super(HQNeuralNet, self).__init__()
        
        # Define the feedforward neural network (fnn)
        self.fnn = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, (q_depth+1)*n_qubits),
        )
        self.fnn.apply(init_weights)

        #Define the feedforward (n_qubits, output_size) layer as a output layer
        self.output_layer = nn.Linear(n_qubits, output_size)
        self.output_layer.apply(init_weights)

        # Set the number qubits, number of quantum layers
        self.n_qubits = n_qubits
        self.q_depth = q_depth
        
        # Initialize parameters of quantum circuit
        self.q_weights = nn.Parameter(0.2*torch.randn(q_depth * n_qubits).view(q_depth, n_qubits))

    def forward(self, x):
        """
        Forward pass through the neural network.

        Args:
            data (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        # Pass the data through the feedforward neural network (self.fnn)
        out = self.fnn(x)
        # Initialize an empty tensor for quantum output
        q_out = torch.Tensor(0, self.n_qubits)
        
        for elem in out:
            # Apply the quantum circuit (hquantum_net) to each output element
            elem = elem.view((self.q_depth+1), self.n_qubits)
            q_out_elem = torch.hstack(tuple(hquantum_net(elem, self.n_qubits, self.q_depth, self.q_weights))).float().unsqueeze(0)
            q_out = torch.cat((q_out, q_out_elem))
            
        # Pass the output of quantum circuit through the feedforward linear layer then combine outputs
        out = self.output_layer(q_out)
        out1 = out[:, 0].view(-1, 1)
        out2 = out[:, 1].view(-1, 1)
        return out1*out2

In [9]:
seeds = [14, 42, 86, 195]
hidden_sizes = [10, 30, 50]

q_depth = 3
input_size, output_size = 1, 2

# Store results
results = []

for hs in hidden_sizes:
    mses = []

    for seed in seeds:
        # Initialize and load model
        model_path = f"..\\trained_models\\hybrid damped harmonic oscillator\\{seed}\\hybrid_model_{seed}_{hs}.pth"
        model = HQNeuralNet(input_size, hs, n_qubits, q_depth, output_size)
        model.load_state_dict(torch.load(model_path))
        model.eval()

        # Evaluate model
        with torch.no_grad():
            mse = test_PINN(model, domain, params, num_test)
            mses.append(mse.item())

    # Compute statistics
    mean_mse = np.mean(mses)
    std_mse = np.std(mses)

    # Store result
    results.append([hs, mean_mse, std_mse])

# Create table
df = pd.DataFrame(results, columns=["Hidden Size", "Mean MSE", "Std. Dev. MSE"])

# Format in scientific notation
df["Mean MSE"] = df["Mean MSE"].map(lambda x: f"{x:.5e}")
df["Std. Dev. MSE"] = df["Std. Dev. MSE"].map(lambda x: f"{x:.5e}")

print(df.to_string(index=False))


 Hidden Size    Mean MSE Std. Dev. MSE
          10 1.55672e-07   1.33957e-07
          30 1.21697e-07   1.31893e-07
          50 8.51651e-08   4.21140e-08
